[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YamKhoonTor/datums-book/blob/main/notebooks/Chapter_11_Cross-nation_datum_transformation.ipynb)

Companion notebook to *Datums, Coordinate Systems and Map Projections* by Yam Khoon Tor. Choose *Runtime → Run all*, compare with the output stored beneath each listing, then change the values in the *Try it* cell and run it again.

# Chapter 11. Cross-nation datum transformation

Companion notebook to *Datums, Coordinate Systems and Map Projections: A Practical Guide for Surveyors and Engineers, with SVY21 Worked Examples*, by Yam Khoon Tor.

*Copyright © 2026 Yam Khoon Tor. All rights reserved. Readers who have bought the book may run, copy and modify this notebook for personal study and their own professional work; it may not be republished, distributed or sold without the author's written permission. Provided as a teaching aid, without warranty: verify results independently before use in real work.*

**How to use this notebook.** Run the cells in order from the top: select a cell and press the run button (or Shift+Enter), or choose *Run all* from the menu. Each listing is exactly as printed in the book. Beneath it the notebook stores the output of the run, and repeats the output printed in the book so that the two can be compared. If an unaltered notebook gives different numbers, something is wrong with the installation and should be put right before going on. The last section, **Try it**, repeats an example with the input values gathered at the top of the cell, to be changed freely.

Requirements: Python 3.9 or later and NumPy.

## Set-up: functions from earlier chapters

This chapter reuses functions defined in Listings 2.1, F.1, 3.1, 13.1, 14.1, 14.2. The cell below loads them, so that this notebook can be run on its own.

In [1]:
# Functions from earlier chapters, exactly as printed in the book.
# Run this cell once before the others.


# ---------------- from Listing 2.1: Ellipsoid constants and radii of curvature ----------------
import math

# Defining parameters: semi-major axis a (m) and reciprocal flattening 1/f
ELLIPSOIDS = {
    "WGS84": (6378137.0, 298.257223563),
    "GRS80": (6378137.0, 298.257222101),
    "Everest 1830 (Modified)": (6377304.063, 300.8017),
}


def ellipsoid_constants(a, inv_f):
    """Derived constants of an ellipsoid from a and 1/f."""
    f = 1.0 / inv_f
    b = a * (1.0 - f)
    e2 = 2.0 * f - f * f          # first eccentricity squared
    ep2 = e2 / (1.0 - e2)         # second eccentricity squared
    n = f / (2.0 - f)             # third flattening
    return {"a": a, "f": f, "b": b, "e2": e2, "ep2": ep2, "n": n}


def dms_to_deg(d, m, s):
    """Degrees, minutes, seconds to decimal degrees."""
    sign = -1.0 if d < 0 else 1.0
    return sign * (abs(d) + m / 60.0 + s / 3600.0)


def radii_of_curvature(lat_deg, a, inv_f):
    """Meridian radius M, prime vertical radius nu, mean radius R."""
    c = ellipsoid_constants(a, inv_f)
    phi = math.radians(lat_deg)
    w2 = 1.0 - c["e2"] * math.sin(phi) ** 2
    nu = a / math.sqrt(w2)
    M = a * (1.0 - c["e2"]) / w2 ** 1.5
    R = math.sqrt(M * nu)
    return M, nu, R


# ---------------- from Listing F.1: Utility functions ----------------
import math


def deg_to_dms(x, places=4):
    """Format decimal degrees as degrees, minutes and seconds."""
    sign = "-" if x < 0 else ""
    x = abs(x)
    t = round(x * 3600, places)              # total seconds, rounded once
    d = int(t // 3600)
    m = int((t - d * 3600) // 60)
    s = t - d * 3600 - m * 60
    return f"{sign}{d}° {m:02d}' {s:0{3 + places}.{places}f}\""


def vincenty_inverse(lat1, lon1, lat2, lon2, a, inv_f):
    """Geodesic distance (m) and forward azimuths at both ends (degrees)."""
    c = ellipsoid_constants(a, inv_f)
    f, b = c["f"], c["b"]
    U1 = math.atan((1 - f) * math.tan(math.radians(lat1)))
    U2 = math.atan((1 - f) * math.tan(math.radians(lat2)))
    L = math.radians(lon2 - lon1)
    lam = L
    for _ in range(100):
        sl, cl = math.sin(lam), math.cos(lam)
        ss = math.hypot(math.cos(U2) * sl,
                        math.cos(U1) * math.sin(U2)
                        - math.sin(U1) * math.cos(U2) * cl)
        cs = math.sin(U1) * math.sin(U2) + math.cos(U1) * math.cos(U2) * cl
        sig = math.atan2(ss, cs)
        sa = math.cos(U1) * math.cos(U2) * sl / ss
        c2a = 1 - sa * sa
        c2sm = cs - 2 * math.sin(U1) * math.sin(U2) / c2a if c2a else 0.0
        C = f / 16 * c2a * (4 + f * (4 - 3 * c2a))
        lam_new = L + (1 - C) * f * sa * (
            sig + C * ss * (c2sm + C * cs * (-1 + 2 * c2sm ** 2)))
        converged = abs(lam_new - lam) < 1e-13
        lam = lam_new
        if converged:
            break
    u2 = c2a * (a * a - b * b) / (b * b)
    A = 1 + u2 / 16384 * (4096 + u2 * (-768 + u2 * (320 - 175 * u2)))
    B = u2 / 1024 * (256 + u2 * (-128 + u2 * (74 - 47 * u2)))
    ds = B * ss * (c2sm + B / 4 * (cs * (-1 + 2 * c2sm ** 2)
                                    - B / 6 * c2sm * (-3 + 4 * ss ** 2)
                                    * (-3 + 4 * c2sm ** 2)))
    s = b * A * (sig - ds)
    sl, cl = math.sin(lam), math.cos(lam)
    az1 = math.degrees(math.atan2(
        math.cos(U2) * sl,
        math.cos(U1) * math.sin(U2) - math.sin(U1) * math.cos(U2) * cl)) % 360
    az2 = math.degrees(math.atan2(
        math.cos(U1) * sl,
        -math.sin(U1) * math.cos(U2) + math.cos(U1) * math.sin(U2) * cl)) % 360
    return s, az1, az2


# ---------------- from Listing 3.1: Conversion between geodetic and Cartesian coordinates ----------------
import math


def geodetic_to_ecef(lat_deg, lon_deg, h, a, inv_f):
    """Latitude, longitude (degrees) and height (m) to X, Y, Z (m)."""
    e2 = ellipsoid_constants(a, inv_f)["e2"]
    phi, lam = math.radians(lat_deg), math.radians(lon_deg)
    nu = a / math.sqrt(1.0 - e2 * math.sin(phi) ** 2)
    X = (nu + h) * math.cos(phi) * math.cos(lam)
    Y = (nu + h) * math.cos(phi) * math.sin(lam)
    Z = (nu * (1.0 - e2) + h) * math.sin(phi)
    return X, Y, Z


def ecef_to_geodetic(X, Y, Z, a, inv_f, tol=1e-12):
    """X, Y, Z (m) to latitude, longitude (degrees) and height (m)."""
    e2 = ellipsoid_constants(a, inv_f)["e2"]
    lam = math.atan2(Y, X)
    p = math.hypot(X, Y)
    phi = math.atan2(Z, p * (1.0 - e2))          # first approximation
    for _ in range(20):
        nu = a / math.sqrt(1.0 - e2 * math.sin(phi) ** 2)
        phi_new = math.atan2(Z + e2 * nu * math.sin(phi), p)
        converged = abs(phi_new - phi) < tol
        phi = phi_new
        if converged:
            break
    nu = a / math.sqrt(1.0 - e2 * math.sin(phi) ** 2)
    if abs(math.cos(phi)) > 1e-3:
        h = p / math.cos(phi) - nu
    else:                                        # close to a pole
        h = Z / math.sin(phi) - nu * (1.0 - e2)
    return math.degrees(phi), math.degrees(lam), h


# ---------------- from Listing 13.1: Transverse Mercator forward, inverse, scale factor and convergence (Krüger series) ----------------
import math


def _kruger(a, inv_f):
    """Constants of the Kruger series for an ellipsoid."""
    f = 1.0 / inv_f
    n = f / (2.0 - f)
    B = a / (1.0 + n) * (1.0 + n ** 2 / 4.0 + n ** 4 / 64.0 + n ** 6 / 256.0)
    alpha = (n / 2 - 2 * n ** 2 / 3 + 5 * n ** 3 / 16 + 41 * n ** 4 / 180,
             13 * n ** 2 / 48 - 3 * n ** 3 / 5 + 557 * n ** 4 / 1440,
             61 * n ** 3 / 240 - 103 * n ** 4 / 140,
             49561 * n ** 4 / 161280)
    beta = (n / 2 - 2 * n ** 2 / 3 + 37 * n ** 3 / 96 - n ** 4 / 360,
            n ** 2 / 48 + n ** 3 / 15 - 437 * n ** 4 / 1440,
            17 * n ** 3 / 480 - 37 * n ** 4 / 840,
            4397 * n ** 4 / 161280)
    e = math.sqrt(2.0 * f - f * f)
    return B, alpha, beta, e


def _xi_eta(lat_deg, dlon_rad, e):
    """Coordinates xi', eta' on the conformal sphere, and the auxiliary t."""
    phi = math.radians(lat_deg)
    t = math.sinh(math.atanh(math.sin(phi)) - e * math.atanh(e * math.sin(phi)))
    xi = math.atan2(t, math.cos(dlon_rad))
    eta = math.atanh(math.sin(dlon_rad) / math.sqrt(1.0 + t * t))
    return xi, eta, t


def tm_forward(lat_deg, lon_deg, lat0, lon0, k0, FN, FE, a, inv_f):
    """Latitude, longitude (degrees) to northing, easting (m)."""
    B, alpha, beta, e = _kruger(a, inv_f)
    xi, eta, t = _xi_eta(lat_deg, math.radians(lon_deg - lon0), e)
    X = xi + sum(alpha[j] * math.sin(2 * (j + 1) * xi)
                 * math.cosh(2 * (j + 1) * eta) for j in range(4))
    Y = eta + sum(alpha[j] * math.cos(2 * (j + 1) * xi)
                  * math.sinh(2 * (j + 1) * eta) for j in range(4))
    xi0, _, _ = _xi_eta(lat0, 0.0, e)
    X0 = xi0 + sum(alpha[j] * math.sin(2 * (j + 1) * xi0) for j in range(4))
    return FN + k0 * B * (X - X0), FE + k0 * B * Y


def tm_inverse(N, E, lat0, lon0, k0, FN, FE, a, inv_f):
    """Northing, easting (m) to latitude, longitude (degrees)."""
    B, alpha, beta, e = _kruger(a, inv_f)
    xi0, _, _ = _xi_eta(lat0, 0.0, e)
    X0 = xi0 + sum(alpha[j] * math.sin(2 * (j + 1) * xi0) for j in range(4))
    X = (N - FN) / (k0 * B) + X0
    Y = (E - FE) / (k0 * B)
    xi = X - sum(beta[j] * math.sin(2 * (j + 1) * X)
                 * math.cosh(2 * (j + 1) * Y) for j in range(4))
    eta = Y - sum(beta[j] * math.cos(2 * (j + 1) * X)
                  * math.sinh(2 * (j + 1) * Y) for j in range(4))
    chi = math.asin(math.sin(xi) / math.cosh(eta))        # conformal latitude
    dlon = math.atan2(math.sinh(eta), math.cos(xi))
    phi = chi                                     # iterate to geodetic latitude
    for _ in range(10):
        phi_new = math.asin(math.tanh(math.atanh(math.sin(chi))
                                      + e * math.atanh(e * math.sin(phi))))
        converged = abs(phi_new - phi) < 1e-14
        phi = phi_new
        if converged:
            break
    return math.degrees(phi), lon0 + math.degrees(dlon)


def tm_scale_convergence(lat_deg, lon_deg, lon0, k0, a, inv_f):
    """Point scale factor and grid convergence (degrees)."""
    B, alpha, beta, e = _kruger(a, inv_f)
    f = 1.0 / inv_f
    n = f / (2.0 - f)
    dl = math.radians(lon_deg - lon0)
    xi, eta, t = _xi_eta(lat_deg, dl, e)
    sigma = 1.0 + sum(2 * (j + 1) * alpha[j] * math.cos(2 * (j + 1) * xi)
                      * math.cosh(2 * (j + 1) * eta) for j in range(4))
    tau = sum(2 * (j + 1) * alpha[j] * math.sin(2 * (j + 1) * xi)
              * math.sinh(2 * (j + 1) * eta) for j in range(4))
    phi = math.radians(lat_deg)
    q = (1.0 - n) / (1.0 + n) * math.tan(phi)
    k = k0 * B / a * math.sqrt((1.0 + q ** 2) * (sigma ** 2 + tau ** 2)
                               / (t * t + math.cos(dl) ** 2))
    gamma = math.atan2(tau * math.sqrt(1.0 + t * t) + sigma * t * math.tan(dl),
                       sigma * math.sqrt(1.0 + t * t) - tau * t * math.tan(dl))
    return k, math.degrees(gamma)


# ---------------- from Listing 14.1: Cassini-Soldner projection ----------------
import math


def meridian_arc(lat_deg, a, inv_f):
    """Length of the meridian from the equator to the given latitude (m)."""
    e2 = ellipsoid_constants(a, inv_f)["e2"]
    p = math.radians(lat_deg)
    e4, e6 = e2 * e2, e2 ** 3
    return a * ((1 - e2 / 4 - 3 * e4 / 64 - 5 * e6 / 256) * p
                - (3 * e2 / 8 + 3 * e4 / 32 + 45 * e6 / 1024) * math.sin(2 * p)
                + (15 * e4 / 256 + 45 * e6 / 1024) * math.sin(4 * p)
                - (35 * e6 / 3072) * math.sin(6 * p))


def footpoint_latitude(m1, a, inv_f):
    """Latitude (radians) at which the meridian arc from the equator is m1."""
    e2 = ellipsoid_constants(a, inv_f)["e2"]
    e4, e6 = e2 * e2, e2 ** 3
    mu = m1 / (a * (1 - e2 / 4 - 3 * e4 / 64 - 5 * e6 / 256))
    e1 = (1 - math.sqrt(1 - e2)) / (1 + math.sqrt(1 - e2))
    return (mu + (3 * e1 / 2 - 27 * e1 ** 3 / 32) * math.sin(2 * mu)
            + (21 * e1 ** 2 / 16 - 55 * e1 ** 4 / 32) * math.sin(4 * mu)
            + (151 * e1 ** 3 / 96) * math.sin(6 * mu)
            + (1097 * e1 ** 4 / 512) * math.sin(8 * mu))


def cassini_forward(lat_deg, lon_deg, lat0, lon0, FN, FE, a, inv_f):
    """Cassini-Soldner forward: latitude, longitude to northing, easting."""
    c = ellipsoid_constants(a, inv_f)
    e2, ep2 = c["e2"], c["ep2"]
    p = math.radians(lat_deg)
    nu = a / math.sqrt(1 - e2 * math.sin(p) ** 2)
    A = math.radians(lon_deg - lon0) * math.cos(p)
    T = math.tan(p) ** 2
    C = ep2 * math.cos(p) ** 2
    E = FE + nu * (A - T * A ** 3 / 6 - (8 - T + 8 * C) * T * A ** 5 / 120)
    N = (FN + meridian_arc(lat_deg, a, inv_f) - meridian_arc(lat0, a, inv_f)
         + nu * math.tan(p) * (A ** 2 / 2 + (5 - T + 6 * C) * A ** 4 / 24))
    return N, E


def cassini_inverse(N, E, lat0, lon0, FN, FE, a, inv_f):
    """Cassini-Soldner inverse: northing, easting to latitude, longitude."""
    e2 = ellipsoid_constants(a, inv_f)["e2"]
    p1 = footpoint_latitude(meridian_arc(lat0, a, inv_f) + (N - FN), a, inv_f)
    nu1 = a / math.sqrt(1 - e2 * math.sin(p1) ** 2)
    M1 = a * (1 - e2) / (1 - e2 * math.sin(p1) ** 2) ** 1.5
    T1 = math.tan(p1) ** 2
    D = (E - FE) / nu1
    phi = p1 - (nu1 * math.tan(p1) / M1) * (D ** 2 / 2 - (1 + 3 * T1) * D ** 4 / 24)
    lam = math.radians(lon0) + (D - T1 * D ** 3 / 3
                                + (1 + 3 * T1) * T1 * D ** 5 / 15) / math.cos(p1)
    return math.degrees(phi), math.degrees(lam)


# ---------------- from Listing 14.2: Hotine oblique Mercator projection, variant A ----------------
def hom_setup(latc, lonc, azc, gammac, kc, FE, FN, a, inv_f):
    """Derived constants of the oblique Mercator from its defining parameters.

    latc, lonc: projection centre; azc: azimuth of the central line;
    gammac: angle from rectified to skew grid (all in degrees);
    kc: scale factor on the central line; FE, FN: false easting and northing.
    """
    e2 = ellipsoid_constants(a, inv_f)["e2"]
    e = math.sqrt(e2)
    pc, lc = math.radians(latc), math.radians(lonc)
    ac, gc = math.radians(azc), math.radians(gammac)
    B = math.sqrt(1 + e2 * math.cos(pc) ** 4 / (1 - e2))
    A = a * B * kc * math.sqrt(1 - e2) / (1 - e2 * math.sin(pc) ** 2)
    t0 = math.tan(math.pi / 4 - pc / 2) / (
        (1 - e * math.sin(pc)) / (1 + e * math.sin(pc))) ** (e / 2)
    D = B * math.sqrt(1 - e2) / (math.cos(pc)
                                 * math.sqrt(1 - e2 * math.sin(pc) ** 2))
    D2 = max(D * D, 1.0)
    F = D + math.sqrt(D2 - 1) * (1 if latc >= 0 else -1)
    H = F * t0 ** B
    G = (F - 1 / F) / 2
    g0 = math.asin(math.sin(ac) / D)
    l0 = lc - math.asin(G * math.tan(g0)) / B
    return dict(e=e, A=A, B=B, H=H, g0=g0, l0=l0, gc=gc, FE=FE, FN=FN)


def hom_forward(lat_deg, lon_deg, P):
    """Oblique Mercator forward: latitude, longitude to northing, easting."""
    e, A, B, H, g0, l0, gc = (P[k] for k in
                              ("e", "A", "B", "H", "g0", "l0", "gc"))
    p, l = math.radians(lat_deg), math.radians(lon_deg)
    t = math.tan(math.pi / 4 - p / 2) / (
        (1 - e * math.sin(p)) / (1 + e * math.sin(p))) ** (e / 2)
    Q = H / t ** B
    S, T = (Q - 1 / Q) / 2, (Q + 1 / Q) / 2
    V = math.sin(B * (l - l0))
    U = (-V * math.cos(g0) + S * math.sin(g0)) / T
    v = A * math.log((1 - U) / (1 + U)) / (2 * B)        # across the central line
    u = A * math.atan2(S * math.cos(g0) + V * math.sin(g0),
                       math.cos(B * (l - l0))) / B      # along the central line
    E = v * math.cos(gc) + u * math.sin(gc) + P["FE"]    # rectify
    N = u * math.cos(gc) - v * math.sin(gc) + P["FN"]
    return N, E


def hom_inverse(N, E, P):
    """Oblique Mercator inverse: northing, easting to latitude, longitude."""
    e, A, B, H, g0, l0, gc = (P[k] for k in
                              ("e", "A", "B", "H", "g0", "l0", "gc"))
    dE, dN = E - P["FE"], N - P["FN"]
    v = dE * math.cos(gc) - dN * math.sin(gc)
    u = dN * math.cos(gc) + dE * math.sin(gc)
    Q = math.exp(-B * v / A)
    S, T = (Q - 1 / Q) / 2, (Q + 1 / Q) / 2
    V = math.sin(B * u / A)
    U = (V * math.cos(g0) + S * math.sin(g0)) / T
    t = (H / math.sqrt((1 + U) / (1 - U))) ** (1 / B)
    chi = math.pi / 2 - 2 * math.atan(t)                 # conformal latitude
    e2 = e * e
    c2 = e2 / 2 + 5 * e2 ** 2 / 24 + e2 ** 3 / 12 + 13 * e2 ** 4 / 360
    c4 = 7 * e2 ** 2 / 48 + 29 * e2 ** 3 / 240 + 811 * e2 ** 4 / 11520
    c6 = 7 * e2 ** 3 / 120 + 81 * e2 ** 4 / 1120
    c8 = 4279 * e2 ** 4 / 161280
    phi = (chi + c2 * math.sin(2 * chi) + c4 * math.sin(4 * chi)
           + c6 * math.sin(6 * chi) + c8 * math.sin(8 * chi))
    lam = l0 - math.atan2(S * math.cos(g0) - V * math.sin(g0),
                          math.cos(B * u / A)) / B
    return math.degrees(phi), math.degrees(lam)


print("Set-up complete.")

Set-up complete.


## Listing 11.1  From a common campaign to SVY21, Johor Cassini and Peninsula RSO coordinates

This listing is printed in Chapter 11 but uses the projection functions of Chapters 13 and 14, which the set-up cell has loaded. The campaign data are illustrative.

In [2]:
import numpy as np

# Campaign results: name, X Y Z in ITRF2020 at the campaign epoch,
# then X Y Z on the national datum (metres). Illustrative.
SG_MARKS = [
    ("S1", -1519312.318, 6192529.847, 158658.707,
     -1519311.421, 6192530.074, 158658.955),
    ("S2", -1526157.870, 6191007.369, 152579.055,
     -1526156.977, 6191007.595, 152579.301),
    ("S3", -1508016.193, 6195528.969, 148820.219,
     -1508015.296, 6195529.199, 148820.466),
]
MY_MARKS = [
    ("M1", -1510079.069, 6194593.289, 165843.633,
     -1510078.341, 6194593.487, 165843.780),
    ("M2", -1526259.650, 6190490.385, 171370.860,
     -1526258.919, 6190490.580, 171371.006),
    ("M3", -1501350.313, 6196373.787, 179108.636,
     -1501349.579, 6196373.976, 179108.778),
]
BRIDGE = ("S", -1517613.755, 6192871.663, 160623.288)   # ITRF2020, campaign epoch

# SVY21 projection: origin latitude, longitude, k0, false northing, false easting
SVY21_TM = (dms_to_deg(1, 22, 0), dms_to_deg(103, 50, 0), 1.0, 38744.572, 28001.642)

# Johor geocentric Cassini origin, GDM2000 (JUPEM, 2009a)
JOHOR_ORIGIN = (dms_to_deg(2, 2, 33.20196), dms_to_deg(103, 33, 39.83730))


def local_shift(marks):
    """Mean shift (national minus campaign) and the scatter about it (m)."""
    d = np.array([np.array(m[4:7]) - np.array(m[1:4]) for m in marks])
    mean = d.mean(axis=0)
    return mean, d - mean


if __name__ == "__main__":
    shift_sg, scatter_sg = local_shift(SG_MARKS)
    shift_my, scatter_my = local_shift(MY_MARKS)
    print("shift to SVY21  :", np.round(shift_sg, 4),
          " max scatter", f"{np.abs(scatter_sg).max() * 1000:.1f} mm")
    print("shift to GDM2000:", np.round(shift_my, 4),
          " max scatter", f"{np.abs(scatter_my).max() * 1000:.1f} mm")
    print("GDM2000 - SVY21 :", np.round(shift_my - shift_sg, 4))

    XS = np.array(BRIDGE[1:4])
    lat_s, lon_s, h_s = ecef_to_geodetic(*(XS + shift_sg), *ELLIPSOIDS["WGS84"])
    lat_m, lon_m, h_m = ecef_to_geodetic(*(XS + shift_my), *ELLIPSOIDS["GRS80"])
    print("S on SVY21  :", deg_to_dms(lat_s, 5), deg_to_dms(lon_s, 5), f"{h_s:.3f}")
    print("S on GDM2000:", deg_to_dms(lat_m, 5), deg_to_dms(lon_m, 5), f"{h_m:.3f}")

    N, E = tm_forward(lat_s, lon_s, *SVY21_TM, *ELLIPSOIDS["WGS84"])
    print(f"SVY21 grid     N = {N:.3f}  E = {E:.3f}")
    N, E = cassini_forward(lat_m, lon_m, *JOHOR_ORIGIN, 0.0, 0.0,
                           *ELLIPSOIDS["GRS80"])
    print(f"Johor Cassini  N = {N:.3f}  E = {E:.3f}")
    # Peninsula RSO parameters (Table 14.3), as in Listing 14.3
    MRSO = hom_setup(4.0, 102.25, dms_to_deg(323, 1, 32.8458),
                     dms_to_deg(323, 7, 48.3685), 0.99984, 804671.0, 0.0,
                     *ELLIPSOIDS["GRS80"])
    N, E = hom_forward(lat_m, lon_m, MRSO)
    print(f"Peninsula RSO  N = {N:.3f}  E = {E:.3f}")

shift to SVY21  : [0.8957 0.2277 0.247 ]  max scatter 2.7 mm
shift to GDM2000: [0.731 0.194 0.145]  max scatter 5.0 mm
GDM2000 - SVY21 : [-0.1647 -0.0337 -0.102 ]
S on SVY21  : 1° 27' 10.00803" 103° 46' 09.97011" 12.014
S on GDM2000: 1° 27' 10.00470" 103° 46' 09.97554" 12.018
SVY21 grid     N = 48266.651  E = 20890.909
Johor Cassini  N = -65213.712  E = 23188.429
Peninsula RSO  N = 160657.488  E = 641448.684


**Expected output, as printed in the book:**

```
shift to SVY21  : [0.8957 0.2277 0.247 ]  max scatter 2.7 mm
shift to GDM2000: [0.731 0.194 0.145]  max scatter 5.0 mm
GDM2000 - SVY21 : [-0.1647 -0.0337 -0.102 ]
S on SVY21  : 1° 27' 10.00803" 103° 46' 09.97011" 12.014
S on GDM2000: 1° 27' 10.00470" 103° 46' 09.97554" 12.018
SVY21 grid     N = 48266.651  E = 20890.909
Johor Cassini  N = -65213.712  E = 23188.429
Peninsula RSO  N = 160657.488  E = 641448.684
```

## Try it

Change the values between the two dashed lines and run the cell again. Nothing here alters the listings above.

In [3]:
# ---- Change these values, then run the cell ----------------------
# A new point from the same campaign: X, Y, Z in ITRF2020 at the campaign epoch
X_new = (-1517613.755, 6192871.663, 160623.288)
# -------------------------------------------------------------------

shift_sg, _ = local_shift(SG_MARKS)
shift_my, _ = local_shift(MY_MARKS)
X_new = np.array(X_new)

lat_s, lon_s, h_s = ecef_to_geodetic(*(X_new + shift_sg), *ELLIPSOIDS["WGS84"])
N, E = tm_forward(lat_s, lon_s, *SVY21_TM, *ELLIPSOIDS["WGS84"])
print(f"SVY21 grid     N = {N:.3f}  E = {E:.3f}")

lat_m, lon_m, h_m = ecef_to_geodetic(*(X_new + shift_my), *ELLIPSOIDS["GRS80"])
N, E = cassini_forward(lat_m, lon_m, *JOHOR_ORIGIN, 0.0, 0.0, *ELLIPSOIDS["GRS80"])
print(f"Johor Cassini  N = {N:.3f}  E = {E:.3f}")

SVY21 grid     N = 48266.651  E = 20890.909
Johor Cassini  N = -65213.712  E = 23188.429
